# MIMIC-IV-ED -> Triagegeist Schema: Data Integration Notebook

This notebook processes raw MIMIC-IV-ED (v2.2) and MIMIC-IV (v3.1) data
into a single flat CSV aligned with the Triagegeist competition schema (66 columns),
plus MIMIC-specific linkage columns. Every row = one ED visit.

**Key caveats documented throughout:**
- Timestamps are de-identified (shifted ~100 years); day-of-week is unreliable
- `age = anchor_age` is a simplification (true age = anchor_age + (ed_year - anchor_year))
- NEWS2 is approximated (consciousness and supplemental O₂ assumed optimal)
- Comorbidities are inferred from **past** admissions only (leakage-safe)
- 6 schema columns are structurally unavailable in MIMIC (Section 13)

---
## Section 0 - Setup


In [3]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

# --- Paths ---
BASE = Path(".")
RAW_ED   = BASE / "dataset" / "raw" / "mimic-iv-ed" / "ed"
RAW_HOSP = BASE / "dataset" / "raw" / "mimic_iv" / "hosp"
RAW_ICU  = BASE / "dataset" / "raw" / "mimic_iv" / "icu"
OUT_DIR  = BASE / "dataset" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Load target schema ---
with open("dataset/schema_target.json", "r") as f:
    schema_target = json.load(f)

SCHEMA_COLS = list(schema_target["columns"].keys())
print(f"Target schema: {len(SCHEMA_COLS)} columns")
print(SCHEMA_COLS[:10], "...")


Target schema: 66 columns
['patient_id', 'triage_acuity', 'disposition', 'ed_los_hours', 'arrival_mode', 'arrival_day', 'arrival_season', 'shift', 'age_group', 'sex'] ...


## Section 1 - Load raw files

Only the files listed in the task spec are loaded. We select only needed
columns and parse timestamp columns upfront to keep memory usage reasonable.


In [4]:
# --- MIMIC-IV-ED tables ---
edstays = pd.read_csv(
    RAW_ED / "edstays.csv.gz",
    usecols=["subject_id", "hadm_id", "stay_id", "intime", "outtime",
             "gender", "race", "arrival_transport", "disposition"],
    parse_dates=["intime", "outtime"],
)
print(f"edstays: {edstays.shape}")

triage = pd.read_csv(
    RAW_ED / "triage.csv.gz",
    usecols=["subject_id", "stay_id", "temperature", "heartrate", "resprate",
             "o2sat", "sbp", "dbp", "pain", "acuity", "chiefcomplaint"],
)
print(f"triage:  {triage.shape}")

medrecon = pd.read_csv(
    RAW_ED / "medrecon.csv.gz",
    usecols=["subject_id", "stay_id", "name"],
)
print(f"medrecon: {medrecon.shape}")

# --- MIMIC-IV hosp tables ---
patients = pd.read_csv(
    RAW_HOSP / "patients.csv.gz",
    usecols=["subject_id", "gender", "anchor_age", "anchor_year",
             "anchor_year_group", "dod"],
    parse_dates=["dod"],
)
print(f"patients: {patients.shape}")

admissions = pd.read_csv(
    RAW_HOSP / "admissions.csv.gz",
    usecols=["subject_id", "hadm_id", "admittime", "dischtime",
             "insurance", "language"],
    parse_dates=["admittime", "dischtime"],
)
print(f"admissions: {admissions.shape}")

diagnoses_icd = pd.read_csv(
    RAW_HOSP / "diagnoses_icd.csv.gz",
    usecols=["subject_id", "hadm_id", "seq_num", "icd_code", "icd_version"],
)
print(f"diagnoses_icd: {diagnoses_icd.shape}")

omr = pd.read_csv(
    RAW_HOSP / "omr.csv.gz",
    usecols=["subject_id", "chartdate", "result_name", "result_value"],
    parse_dates=["chartdate"],
)
print(f"omr: {omr.shape}")

# --- MIMIC-IV icu tables ---
icustays = pd.read_csv(
    RAW_ICU / "icustays.csv.gz",
    usecols=["subject_id", "hadm_id", "stay_id", "intime", "outtime"],
    parse_dates=["intime", "outtime"],
)
print(f"icustays: {icustays.shape}")


edstays: (425087, 9)
triage:  (425087, 11)
medrecon: (2987342, 3)
patients: (364627, 6)
admissions: (546028, 6)
diagnoses_icd: (6364488, 5)
omr: (7753027, 4)
icustays: (94458, 5)


## Section 2 - Build base cohort

Start from `edstays`, inner-merge with `triage` on `stay_id`.
Keep only rows with valid acuity ∈ [1, 5]. Report dropped rows.


In [5]:
n_edstays = len(edstays)
n_triage = len(triage)

# Inner merge: only keep ED visits that have triage data
df = edstays.merge(triage, on=["subject_id", "stay_id"], how="inner").copy()
print(f"After inner merge edstays×triage: {df.shape[0]:,} rows")
print(f"  Dropped (no triage): {n_edstays - df.shape[0]:,}")

# Filter valid acuity 1-5
df["acuity"] = pd.to_numeric(df["acuity"], errors="coerce")
valid_mask = df["acuity"].between(1, 5)
n_invalid = (~valid_mask).sum()
df = df[valid_mask].copy()
df["triage_acuity"] = df["acuity"].astype(int)
df.drop(columns=["acuity"], inplace=True)

print(f"After acuity filter [1,5]: {df.shape[0]:,} rows")
print(f"  Dropped (invalid acuity): {n_invalid:,}")
print(f"\nAcuity distribution:\n{df['triage_acuity'].value_counts().sort_index()}")


After inner merge edstays×triage: 425,087 rows
  Dropped (no triage): 0
After acuity filter [1,5]: 418,100 rows
  Dropped (invalid acuity): 6,987

Acuity distribution:
triage_acuity
1     24019
2    139411
3    225066
4     28504
5      1100
Name: count, dtype: int64


## Section 3 - Timestamps & year_offset

MIMIC timestamps are de-identified: shifted by a random offset per patient
(approximately 100 years). We compute `year_offset` from the midpoint of
`anchor_year_group` so we can derive a corrected `arrival_year_real`.
We do **not** correct the raw timestamps (keep them shifted for traceability).

**Caveat:** Day-of-week derived from shifted timestamps is unreliable
(the shift is not necessarily a multiple of 7 days).


In [6]:
# Merge patients to get anchor_year, anchor_year_group
df = df.merge(
    patients[["subject_id", "anchor_age", "anchor_year", "anchor_year_group"]],
    on="subject_id", how="left",
).copy()
print(f"After patients merge: {df.shape}")

# Compute year_offset: midpoint of anchor_year_group - anchor_year
MIDPOINTS = {
    "2008 - 2010": 2009,
    "2011 - 2013": 2012,
    "2014 - 2016": 2015,
    "2017 - 2019": 2018,
    "2020 - 2022": 2021,
}
df["year_offset"] = (
    df["anchor_year_group"].map(MIDPOINTS) - df["anchor_year"]
).astype("Int64")

# Derive time features from intime (shifted timestamps)
df["arrival_hour"] = df["intime"].dt.hour
df["arrival_month"] = df["intime"].dt.month
df["arrival_year"] = df["intime"].dt.year  # shifted year

# arrival_day: NOTE - unreliable due to de-identification shift
df["arrival_day"] = df["intime"].dt.day_name()

# arrival_week (ISO week number)
df["arrival_week"] = df["intime"].dt.isocalendar().week.astype("Int64")

# Season based on month (valid despite year shift - shift is in whole years)
season_map = {12: "winter", 1: "winter", 2: "winter",
              3: "spring", 4: "spring", 5: "spring",
              6: "summer", 7: "summer", 8: "summer",
              9: "autumn", 10: "autumn", 11: "autumn"}
df["arrival_season"] = df["arrival_month"].map(season_map)

# Shift (time-of-day bucket)
def hour_to_shift(h):
    if 7 <= h < 13:   return "morning"
    if 13 <= h < 19:  return "afternoon"
    if 19 <= h < 23:  return "evening"
    return "night"

df["shift"] = df["arrival_hour"].apply(hour_to_shift)

# ED length of stay in hours
df["ed_los_hours"] = (
    (df["outtime"] - df["intime"]).dt.total_seconds() / 3600
).round(2)

# Corrected arrival year
df["arrival_year_real"] = df["arrival_year"] + df["year_offset"]

# Rename raw timestamps to MIMIC-specific names
df.rename(columns={"intime": "ed_intime", "outtime": "ed_outtime"}, inplace=True)

print(f"Shape: {df.shape}")
print(f"year_offset distribution:\n{df['year_offset'].value_counts().sort_index()}")
print(f"\narrival_year_real range: {df['arrival_year_real'].min()} – {df['arrival_year_real'].max()}")


After patients merge: (418100, 21)
Shape: (418100, 31)
year_offset distribution:
year_offset
-194      18
-193      65
-192     370
-191     806
-190     810
        ... 
-96     1724
-95     1924
-94     1031
-93      931
-92      885
Name: count, Length: 103, dtype: Int64

arrival_year_real range: 2009 – 2021


## Section 4 - Demographics

- **age**: Uses `anchor_age` from the patients table. This is the patient's age
  at `anchor_year`, not at the ED visit. True age = anchor_age + (ed_year - anchor_year),
  but since we don't have the true ed_year for all patients, we use anchor_age
  as a simplification. For patients with anchor_age=91, MIMIC caps age at 91.
- **sex**: From `patients.gender` (M/F).
- **insurance_type**: From `admissions.insurance` (mapped to schema categories).
- **language**: From `admissions.language` (lowercased).
- **age_group**: Derived from age.
- **site_id**: Always 'BIDMC' (Beth Israel Deaconess Medical Center).


In [8]:
# Sex from patients (gender_x is from edstays, but patients.gender is canonical)
# We already merged patients above; gender_y from patients
if "gender_y" in df.columns:
    df["sex"] = df["gender_y"]
    df.drop(columns=[c for c in df.columns if c.startswith("gender")], inplace=True, errors="ignore")
    df["sex"] = df["sex"].where(df["sex"].isin(["M", "F"]))
elif "gender_x" in df.columns:
    df["sex"] = df["gender_x"].where(df["gender_x"].isin(["M", "F"]))
    df.drop(columns=["gender_x"], inplace=True, errors="ignore")
elif "gender" in df.columns:
    df["sex"] = df["gender"].where(df["gender"].isin(["M", "F"]))
    df.drop(columns=["gender"], inplace=True, errors="ignore")

# Age from anchor_age
df["age"] = pd.to_numeric(df["anchor_age"], errors="coerce").astype("Int64")

# Age group
conditions = [
    df["age"] < 18,
    df["age"].between(18, 39),
    df["age"].between(40, 64),
    df["age"] >= 65,
]
choices = ["pediatric", "young_adult", "middle_aged", "senior"]
df["age_group"] = pd.cut(
    df["age"].astype(float),
    bins=[-1, 17, 39, 64, 200],
    labels=["pediatric", "young_adult", "middle_aged", "senior"],
).astype(object)
df.loc[df["age_group"] == "None", "age_group"] = np.nan

# Insurance and language from admissions (left merge on hadm_id)
adm_demo = admissions[["hadm_id", "insurance", "language"]].drop_duplicates("hadm_id")
df = df.merge(adm_demo, on="hadm_id", how="left").copy()
print(f"After admissions merge: {df.shape}")

# Map insurance
INS_MAP = {
    "Medicare": "public",
    "Medicaid": "public",
    "Government": "public",
    "Private": "private",
    "Self Pay": "self-pay",
    "Other": "other",
}
# Defensive: handle case variations
if "insurance" in df.columns:
    df["insurance_type"] = df["insurance"].map(INS_MAP).fillna("other")
    # If hadm_id was NaN (discharged from ED), insurance_type stays NaN
    df.loc[df["hadm_id"].isna(), "insurance_type"] = np.nan
    df.drop(columns=["insurance"], inplace=True, errors="ignore")

# Language: lowercase
if "language" in df.columns:
    df["language"] = df["language"].str.lower()

# Site
df["site_id"] = "BIDMC"

print(f"Shape: {df.shape}")
print(f"Sex: {df['sex'].value_counts().to_dict()}")
print(f"Age group: {df['age_group'].value_counts().to_dict()}")
print(f"Insurance: {df.get('insurance_type', pd.Series()).value_counts().to_dict()}")


After admissions merge: (418100, 35)
Shape: (418100, 36)
Sex: {'F': 227007, 'M': 191093}
Age group: {'middle_aged': 164503, 'young_adult': 142058, 'senior': 111464}
Insurance: {'public': 134469, 'private': 54756, 'other': 8599}


## Section 5 - Triage vitals

Rename MIMIC triage columns to schema names. Convert temperature from
Fahrenheit to Celsius. Parse pain string to numeric `pain_score`.


In [9]:
# Rename vital columns
vital_rename = {
    "sbp": "systolic_bp",
    "dbp": "diastolic_bp",
    "heartrate": "heart_rate",
    "resprate": "respiratory_rate",
    "o2sat": "spo2",
}
for old, new in vital_rename.items():
    if old in df.columns:
        df[new] = pd.to_numeric(df[old], errors="coerce")
        df.drop(columns=[old], inplace=True)

# Temperature: MIMIC triage temperature is in Fahrenheit
if "temperature" in df.columns:
    temp_f = pd.to_numeric(df["temperature"], errors="coerce")
    # Filter plausible F range before converting
    temp_f = temp_f.where(temp_f.between(80, 115))
    df["temperature_c"] = ((temp_f - 32) * 5 / 9).round(1)
    df.drop(columns=["temperature"], inplace=True)

# Pain: MIMIC stores pain as a string (e.g., "5", "0", "Unable to Answer")
if "pain" in df.columns:
    df["pain_score"] = pd.to_numeric(df["pain"], errors="coerce")
    df["pain_score"] = df["pain_score"].where(df["pain_score"].between(0, 10))
    df.drop(columns=["pain"], inplace=True)

# Chief complaint raw
if "chiefcomplaint" in df.columns:
    df.rename(columns={"chiefcomplaint": "chief_complaint_raw"}, inplace=True)

print(f"Shape: {df.shape}")
for v in ["systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate",
           "spo2", "temperature_c", "pain_score"]:
    if v in df.columns:
        pct_miss = df[v].isna().mean() * 100
        print(f"  {v}: median={df[v].median():.1f}, missing={pct_miss:.1f}%")


Shape: (418100, 36)
  systolic_bp: median=133.0, missing=2.7%
  diastolic_bp: median=77.0, missing=2.9%
  heart_rate: median=84.0, missing=2.5%
  respiratory_rate: median=18.0, missing=3.2%
  spo2: median=99.0, missing=3.3%
  temperature_c: median=36.7, missing=4.1%
  pain_score: median=4.0, missing=7.8%


## Section 6 - Derived vitals

Compute MAP, pulse pressure, shock index, and an approximate NEWS2 score.

**NEWS2 approximation:** The Royal College of Physicians (2017) NEWS2 uses 7
parameters. We have 5 (RR, SpO₂, SBP, HR, temperature). We assume:
- Level of consciousness = Alert (score 0)
- Supplemental O₂ = No (score 0)

This systematically underestimates NEWS2 by 0–5 points for patients who are
not alert or receiving supplemental oxygen.


In [10]:
# Mean arterial pressure
if "systolic_bp" in df.columns and "diastolic_bp" in df.columns:
    df["mean_arterial_pressure"] = (
        (df["systolic_bp"] + 2 * df["diastolic_bp"]) / 3
    ).round(1)

# Pulse pressure
if "systolic_bp" in df.columns and "diastolic_bp" in df.columns:
    df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]

# Shock index (clipped to [0.1, 5.0])
if "heart_rate" in df.columns and "systolic_bp" in df.columns:
    si = df["heart_rate"] / df["systolic_bp"]
    df["shock_index"] = si.where(si.between(0.1, 5.0)).round(3)

# --- NEWS2 approximate score (Royal College of Physicians, 2017) ---
def compute_news2_approx(row):
    """
    Compute NEWS2 from 5 available vitals.
    Consciousness (ACVPU) and supplemental O2 are unavailable -> assumed optimal (score=0).
    Each parameter scores 0-3, total range 0-20 (but effectively 0-15 here).
    """
    score = 0
    has_any = False

    # Respiratory rate
    rr = row.get("respiratory_rate", np.nan)
    if pd.notna(rr):
        has_any = True
        if   rr <= 8:   score += 3
        elif rr <= 11:  score += 1
        elif rr <= 20:  score += 0
        elif rr <= 24:  score += 2
        else:           score += 3

    # SpO2 (Scale 1 - standard, not COPD pathway)
    spo2 = row.get("spo2", np.nan)
    if pd.notna(spo2):
        has_any = True
        if   spo2 <= 91:  score += 3
        elif spo2 <= 93:  score += 2
        elif spo2 <= 95:  score += 1
        else:             score += 0

    # Systolic BP
    sbp = row.get("systolic_bp", np.nan)
    if pd.notna(sbp):
        has_any = True
        if   sbp <= 90:   score += 3
        elif sbp <= 100:  score += 2
        elif sbp <= 110:  score += 1
        elif sbp <= 219:  score += 0
        else:             score += 3

    # Heart rate
    hr = row.get("heart_rate", np.nan)
    if pd.notna(hr):
        has_any = True
        if   hr <= 40:   score += 3
        elif hr <= 50:   score += 1
        elif hr <= 90:   score += 0
        elif hr <= 110:  score += 1
        elif hr <= 130:  score += 2
        else:            score += 3

    # Temperature (Celsius)
    temp = row.get("temperature_c", np.nan)
    if pd.notna(temp):
        has_any = True
        if   temp <= 35.0:  score += 3
        elif temp <= 36.0:  score += 1
        elif temp <= 38.0:  score += 0
        elif temp <= 39.0:  score += 1
        else:               score += 2

    return score if has_any else np.nan

print("Computing approximate NEWS2 scores (this may take a minute)...")
required = ["respiratory_rate", "spo2", "systolic_bp", "heart_rate", "temperature_c"]
if all(c in df.columns for c in required):
    df["news2_score"] = df[required].apply(compute_news2_approx, axis=1)
    print(f"  NEWS2 computed: {df['news2_score'].notna().sum():,} rows")
    print(f"  Mean={df['news2_score'].mean():.2f}, Median={df['news2_score'].median():.0f}")
else:
    missing = [c for c in required if c not in df.columns]
    print(f"WARNING: Cannot compute NEWS2 - missing: {missing}")

print(f"Shape: {df.shape}")


Computing approximate NEWS2 scores (this may take a minute)...
  NEWS2 computed: 409,104 rows
  Mean=0.93, Median=1
Shape: (418100, 40)


## Section 7 - Anthropometry from OMR

The `omr` (Outpatient Medical Record) table contains weight, height, and BMI
measurements from clinic visits. For each ED visit, we find the closest OMR
measurement within ±180 days of `ed_intime`. Weight is converted from lbs to kg,
height from inches to cm.


In [12]:
# Parse OMR for relevant measures
omr_filtered = omr[
    omr["result_name"].isin(["Weight (Lbs)", "Height (Inches)", "BMI (kg/m2)"])
].copy()
omr_filtered["result_value"] = pd.to_numeric(omr_filtered["result_value"], errors="coerce")
omr_filtered = omr_filtered.dropna(subset=["result_value"])
print(f"OMR relevant rows: {omr_filtered.shape[0]:,}")

# Pivot to wide: one row per (subject_id, chartdate, measure)
omr_wide = omr_filtered.pivot_table(
    index=["subject_id", "chartdate"],
    columns="result_name",
    values="result_value",
    aggfunc="first",
).reset_index()
omr_wide.columns.name = None

# Rename columns
col_map = {"Weight (Lbs)": "weight_lbs", "Height (Inches)": "height_in", "BMI (kg/m2)": "bmi_omr"}
omr_wide.rename(columns=col_map, inplace=True)
print(f"OMR pivoted: {omr_wide.shape}")

def find_closest_omr(group, ed_times_df, sid):
    """For each ED visit of a patient, find the closest OMR record within ±180 days."""
    if sid not in ed_times_df.index:
        return pd.DataFrame()

    ed_rows = ed_times_df.loc[[sid]]
    results = []

    for _, ed_row in ed_rows.iterrows():
        ed_dt = ed_row["ed_intime"]
        diffs = (group["chartdate"] - ed_dt).abs()
        within = diffs <= pd.Timedelta(days=180)
        if within.any():
            closest_idx = diffs[within].idxmin()
            row = group.loc[closest_idx].copy()
            row["stay_id"] = ed_row["stay_id"]
            results.append(row)

    if results:
        return pd.DataFrame(results)
    return pd.DataFrame()

# Prepare lookup
ed_times = df[["subject_id", "stay_id", "ed_intime"]].set_index("subject_id")
# Only process patients that exist in OMR
common_sids = set(df["subject_id"].unique()) & set(omr_wide["subject_id"].unique())
omr_subset = omr_wide[omr_wide["subject_id"].isin(common_sids)].copy()

print(f"Patients with OMR data: {len(common_sids):,} / {df['subject_id'].nunique():,}")

# Group by subject_id and find closest for each visit
matched = []
for sid, grp in omr_subset.groupby("subject_id"):
    res = find_closest_omr(grp, ed_times, sid)
    if len(res) > 0:
        matched.append(res)

if matched:
    omr_matched = pd.concat(matched, ignore_index=True)
    # Keep only the columns we need
    keep_cols = ["stay_id"]
    if "weight_lbs" in omr_matched.columns:
        keep_cols.append("weight_lbs")
    if "height_in" in omr_matched.columns:
        keep_cols.append("height_in")
    if "bmi_omr" in omr_matched.columns:
        keep_cols.append("bmi_omr")

    omr_matched = omr_matched[keep_cols].drop_duplicates("stay_id")

    # Convert units
    if "weight_lbs" in omr_matched.columns:
        omr_matched["weight_kg"] = (omr_matched["weight_lbs"] * 0.453592).round(1)
    if "height_in" in omr_matched.columns:
        omr_matched["height_cm"] = (omr_matched["height_in"] * 2.54).round(1)

    # BMI: prefer direct if available, else derive
    if "bmi_omr" in omr_matched.columns:
        omr_matched["bmi"] = omr_matched["bmi_omr"].round(1)
    if "weight_kg" in omr_matched.columns and "height_cm" in omr_matched.columns:
        derive_mask = omr_matched["bmi"].isna() if "bmi" in omr_matched.columns else pd.Series(True, index=omr_matched.index)
        h_m = omr_matched["height_cm"] / 100
        derived_bmi = (omr_matched["weight_kg"] / (h_m ** 2)).round(1)
        if "bmi" not in omr_matched.columns:
            omr_matched["bmi"] = derived_bmi
        else:
            omr_matched.loc[derive_mask, "bmi"] = derived_bmi[derive_mask]

    # Keep final columns for merge
    merge_cols = ["stay_id"]
    for c in ["weight_kg", "height_cm", "bmi"]:
        if c in omr_matched.columns:
            merge_cols.append(c)

    omr_final = omr_matched[merge_cols]
    df = df.merge(omr_final, on="stay_id", how="left").copy()
    print(f"Merged OMR. Shape: {df.shape}")
    for c in ["weight_kg", "height_cm", "bmi"]:
        if c in df.columns:
            print(f"  {c}: {df[c].notna().sum():,} non-null ({df[c].notna().mean()*100:.1f}%)")
else:
    print("No OMR matches found within ±180 days window.")
    for c in ["weight_kg", "height_cm", "bmi"]:
        df[c] = np.nan

print(f"Shape: {df.shape}")


OMR relevant rows: 4,861,812
OMR pivoted: (2034552, 5)
Patients with OMR data: 119,323 / 201,252
Merged OMR. Shape: (418100, 43)
  weight_kg: 263,711 non-null (63.1%)
  height_cm: 120,512 non-null (28.8%)
  bmi: 248,151 non-null (59.4%)
Shape: (418100, 43)


## Section 8 - Comorbidities from diagnoses_icd

For each patient, we aggregate ICD-9 and ICD-10 diagnoses from **all past
hospital admissions** where `admittime < ed_intime`. This ensures no data
leakage from the current or future visits. We use all `seq_num` values
(not just primary diagnosis).

The ICD prefix mapping below covers the 25 `hx_*` columns in the schema.


In [ ]:
# --- Import the dedicated ICD mapping module ---
from icd_comorbidity_map import compute_hx_flags

# Build admission timeline for leakage-safe filtering
adm_times = admissions[["subject_id", "hadm_id", "admittime"]].drop_duplicates("hadm_id")

# For each ED visit, find all hadm_ids with admittime STRICTLY before ed_intime
ed_adm = df[["subject_id", "stay_id", "ed_intime"]].merge(
    adm_times, on="subject_id", how="inner"
)
ed_adm = ed_adm[ed_adm["admittime"] < ed_adm["ed_intime"]]
prior_hadm_ids = ed_adm[["stay_id", "hadm_id"]].drop_duplicates()
print(f"Prior (subject, hadm) pairs: {len(prior_hadm_ids):,}")

# Filter diagnoses_icd to only prior admissions, tag each row with stay_id
diag_prior = diagnoses_icd.merge(prior_hadm_ids, on="hadm_id", how="inner")
print(f"Diagnosis rows from prior admissions: {len(diag_prior):,}")

# Compute hx_* flags grouped by stay_id (one row per ED visit)
hx_flags = compute_hx_flags(diag_prior, group_by="stay_id")
print(f"Flags computed for {len(hx_flags):,} stays")

# Merge into main df
hx_cols = [c for c in hx_flags.columns if c.startswith("hx_")] + ["num_comorbidities"]
df = df.merge(hx_flags[["stay_id"] + hx_cols], on="stay_id", how="left").copy()

# Patients without any prior admission → all flags = 0
for col in hx_cols:
    df[col] = df[col].fillna(0).astype(int)

print(f"Shape: {df.shape}")
print(f"Patients with ≥1 comorbidity: {(df['num_comorbidities'] > 0).sum():,} "
      f"({(df['num_comorbidities'] > 0).mean()*100:.1f}%)")
print(f"Mean comorbidities: {df['num_comorbidities'].mean():.2f}")

Diagnoses with admission times: 6,364,488
Building comorbidity lookup from ICD codes...
  hx_hypertension: 111,008 patients, 279,107 diagnosis records
  hx_diabetes_type2: 26,344 patients, 89,083 diagnosis records
  hx_diabetes_type1: 1,762 patients, 11,148 diagnosis records
  hx_asthma: 22,960 patients, 50,328 diagnosis records
  hx_copd: 20,794 patients, 46,931 diagnosis records
  hx_heart_failure: 31,369 patients, 112,936 diagnosis records


KeyboardInterrupt: 

## Section 9 - Medications

From `medrecon` (medication reconciliation at ED triage), count the number
of active medications per stay.


In [ ]:
# Count distinct medications per stay
med_counts = (
    medrecon.groupby("stay_id")["name"]
    .nunique()
    .reset_index()
    .rename(columns={"name": "num_active_medications"})
)
print(f"Medication counts for {len(med_counts):,} stays")

df = df.merge(med_counts, on="stay_id", how="left").copy()
df["num_active_medications"] = df["num_active_medications"].fillna(0).astype(int)

print(f"Shape: {df.shape}")
print(f"Median medications: {df['num_active_medications'].median():.0f}")
print(f"Mean medications: {df['num_active_medications'].mean():.1f}")


## Section 10 - Prior visits (12-month lookback)

Count prior ED visits and prior hospital admissions within 365 days
before `ed_intime`, excluding the current visit/admission.
Uses sort + groupby for performance (no nested loops).


In [ ]:
# --- Prior ED visits in last 12 months ---
ed_visits = df[["subject_id", "stay_id", "ed_intime"]].sort_values(
    ["subject_id", "ed_intime"]
).copy()

def count_prior_ed(group):
    """For each visit, count how many prior visits occurred within 365 days."""
    times = group["ed_intime"].values
    counts = []
    for i in range(len(times)):
        cutoff = times[i] - pd.Timedelta(days=365)
        # Count visits with intime >= cutoff AND intime < current
        n = ((times[:i] >= cutoff) & (times[:i] < times[i])).sum()
        counts.append(n)
    return pd.Series(counts, index=group.index)

print("Counting prior ED visits (12-month window)...")
df["num_prior_ed_visits_12m"] = (
    ed_visits.groupby("subject_id", group_keys=False)
    .apply(count_prior_ed)
    .astype(int)
)
print(f"  Done. Mean: {df['num_prior_ed_visits_12m'].mean():.2f}")

# --- Prior hospital admissions in last 12 months ---
adm_for_prior = admissions[["subject_id", "hadm_id", "admittime"]].copy()

def count_prior_admissions(row, adm_df):
    cutoff = row["ed_intime"] - pd.Timedelta(days=365)
    mask = (
        (adm_df["subject_id"] == row["subject_id"])
        & (adm_df["admittime"] >= cutoff)
        & (adm_df["admittime"] < row["ed_intime"])
    )
    # Exclude current hadm_id if present
    if pd.notna(row.get("hadm_id")):
        mask = mask & (adm_df["hadm_id"] != row["hadm_id"])
    return mask.sum()

# Optimized: pre-group admissions by subject_id
print("Counting prior hospital admissions (12-month window)...")
adm_grouped = adm_for_prior.groupby("subject_id")
prior_adm_counts = []

for _, row in df[["subject_id", "stay_id", "hadm_id", "ed_intime"]].iterrows():
    sid = row["subject_id"]
    if sid in adm_grouped.groups:
        grp = adm_grouped.get_group(sid)
        cutoff = row["ed_intime"] - pd.Timedelta(days=365)
        mask = (grp["admittime"] >= cutoff) & (grp["admittime"] < row["ed_intime"])
        if pd.notna(row["hadm_id"]):
            mask = mask & (grp["hadm_id"] != row["hadm_id"])
        prior_adm_counts.append(mask.sum())
    else:
        prior_adm_counts.append(0)

df["num_prior_admissions_12m"] = prior_adm_counts
print(f"  Done. Mean: {df['num_prior_admissions_12m'].mean():.2f}")
print(f"Shape: {df.shape}")


## Section 11 - Visit number

For each patient, compute chronological visit ordering and total visits.


In [ ]:
df = df.sort_values(["subject_id", "ed_intime"]).copy()

# visit_number: 1-indexed chronological order per patient
df["visit_number"] = df.groupby("subject_id").cumcount() + 1

# total_visits_patient: total ED visits per patient in the cohort
visit_totals = df.groupby("subject_id")["stay_id"].transform("count")
df["total_visits_patient"] = visit_totals.astype(int)

print(f"Shape: {df.shape}")
print(f"Visit number stats:\n{df['visit_number'].describe()}")
print(f"\nPatients with >1 visit: {(df['total_visits_patient'] > 1).sum():,} rows")


## Section 12 - Disposition & arrival_mode

Map MIMIC disposition categories to schema values. Map arrival transport.


In [ ]:
# --- Disposition mapping ---
DISP_MAP = {
    "HOME": "discharged",
    "ADMITTED": "admitted",
    "TRANSFER": "transferred",
    "EXPIRED": "deceased",
    "LEFT WITHOUT BEING SEEN": "lwbs",
    "LEFT AGAINST MEDICAL ADVICE": "lama",
    "ELOPED": "lama",
    "OTHER": "discharged",
}

if "disposition" in df.columns:
    df["disposition"] = (
        df["disposition"].str.upper().str.strip().map(DISP_MAP).fillna("discharged")
    )
    print(f"Disposition distribution:\n{df['disposition'].value_counts()}")

# --- Arrival mode mapping ---
TRANSPORT_MAP = {
    "WALK IN": "walk-in",
    "AMBULANCE": "ambulance",
    "HELICOPTER": "helicopter",
    "OTHER": "other",
    "UNKNOWN": "other",
}

if "arrival_transport" in df.columns:
    df["arrival_mode"] = (
        df["arrival_transport"].str.upper().str.strip().map(TRANSPORT_MAP).fillna("other")
    )
    df.drop(columns=["arrival_transport"], inplace=True)
    print(f"\nArrival mode distribution:\n{df['arrival_mode'].value_counts()}")

print(f"Shape: {df.shape}")


## Section 13 - Unavailable columns as NaN

The following 6 schema columns are **structurally unavailable** in MIMIC-IV-ED
and are set to NaN with documentation:

1. `transport_origin` - not recorded in MIMIC-IV-ED
2. `pain_location` - triage only records chief complaint text, not structured pain site
3. `mental_status_triage` - no ACVPU/mental status at triage in MIMIC-IV-ED
4. `triage_nurse_id` - de-identified; not available
5. `gcs_total` - GCS is recorded in ICU chartevents, not at ED triage
6. `chief_complaint_system` - requires NLP classification (done in separate pipeline)

Note: `data_source` is set to `'mimic_iv_ed'` (not NaN).


In [ ]:
# Structurally unavailable columns - set to NaN with reason
unavailable_cols = {
    "transport_origin": "Not recorded in MIMIC-IV-ED",
    "pain_location": "Triage records free-text chief complaint only, not structured pain site",
    "mental_status_triage": "No ACVPU/mental status at ED triage in MIMIC-IV-ED",
    "triage_nurse_id": "De-identified; not available in MIMIC-IV-ED",
    "gcs_total": "GCS is in ICU chartevents, not available at ED triage",
    "chief_complaint_system": "Requires NLP classification (separate pipeline)",
}

for col, reason in unavailable_cols.items():
    df[col] = np.nan
    print(f"  {col} -> NaN  ({reason})")

# data_source is NOT NaN - it identifies the dataset
df["data_source"] = "mimic_iv_ed"

print(f"\nShape: {df.shape}")


## Section 14 - Outlier handling

Three-pass approach:
- **(A)** Clip to schema-defined ranges; values outside -> NaN
- **(B)** Global Z-score > 5 -> NaN (catches extreme within-range values)
- **(C)** Coefficient of variation warning: flag columns with std/mean > 1.0


In [ ]:
# Build range dict from schema
range_dict = {}
for col_name, spec in schema_target["columns"].items():
    if "range" in spec and spec["range"] is not None:
        lo, hi = spec["range"]
        range_dict[col_name] = (lo, hi)

print(f"Schema ranges defined for {len(range_dict)} columns")

# --- Pass A: Clip to schema ranges ---
print("\n--- Pass A: Schema range clipping ---")
for col, (lo, hi) in range_dict.items():
    if col not in df.columns:
        continue
    series = pd.to_numeric(df[col], errors="coerce")
    outside = series.notna() & ~series.between(lo, hi)
    n_outside = outside.sum()
    if n_outside > 0:
        df[col] = series.where(series.between(lo, hi))
        print(f"  {col}: {n_outside:,} values outside [{lo}, {hi}] -> NaN")

# --- Pass B: Z-score > 5 -> NaN ---
print("\n--- Pass B: Z-score outlier removal (|Z| > 5) ---")
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col in ["subject_id", "stay_id", "hadm_id", "visit_number",
               "total_visits_patient", "year_offset", "arrival_year",
               "arrival_year_real", "triage_acuity"]:
        continue  # skip IDs and ordinal/categorical integers
    series = df[col].dropna()
    if len(series) < 10:
        continue
    mean, std = series.mean(), series.std()
    if std == 0:
        continue
    z = ((df[col] - mean) / std).abs()
    outlier_mask = z > 5
    n_outliers = outlier_mask.sum()
    if n_outliers > 0:
        df.loc[outlier_mask, col] = np.nan
        print(f"  {col}: {n_outliers:,} Z-score outliers -> NaN")

# --- Pass C: Coefficient of variation warning ---
print("\n--- Pass C: Coefficient of variation check (CV > 1.0) ---")
for col in numeric_cols:
    if col in ["subject_id", "stay_id", "hadm_id"]:
        continue
    series = df[col].dropna()
    if len(series) < 10 or series.mean() == 0:
        continue
    cv = series.std() / abs(series.mean())
    if cv > 1.0:
        print(f"  WARNING: {col} has CV={cv:.2f} (std/mean > 1.0)")

print(f"\nShape after outlier handling: {df.shape}")


## Section 15 - Missing flags

For each column with >5% missing values, add a binary `<col>_missing` indicator.


In [ ]:
# Identify columns with >5% missing
missing_pct = df.isna().mean()
high_missing = missing_pct[missing_pct > 0.05].index.tolist()

# Exclude MIMIC-specific extras and already-NaN columns from flag creation
exclude_from_flags = {"transport_origin", "pain_location", "mental_status_triage",
                      "triage_nurse_id", "gcs_total", "chief_complaint_system",
                      "hadm_id", "ed_intime", "ed_outtime", "anchor_year_group",
                      "year_offset", "arrival_year_real", "arrival_week",
                      "data_source", "patient_id"}

flag_cols = [c for c in high_missing if c not in exclude_from_flags]

print(f"Columns with >5% missing (creating flags): {len(flag_cols)}")
for col in sorted(flag_cols):
    flag_name = f"{col}_missing"
    df[flag_name] = df[col].isna().astype(int)
    pct = df[col].isna().mean() * 100
    print(f"  {flag_name}: {pct:.1f}% missing")

print(f"\nShape: {df.shape}")


## Section 16 - Schema alignment

Build the final column order:
1. `patient_id` = `MIMIC_<subject_id>_<stay_id>`
2. Remaining 65 schema columns in `schema_target.json` order
3. MIMIC-specific extras
4. `*_missing` flag columns

Assert all 66 schema columns are present.


In [ ]:
# Build patient_id
df["patient_id"] = (
    "MIMIC_" + df["subject_id"].astype(str) + "_" + df["stay_id"].astype(str)
)

# Drop helper columns no longer needed
drop_cols = ["anchor_age", "anchor_year", "race"]
for c in drop_cols:
    if c in df.columns:
        df.drop(columns=[c], inplace=True)

# Ensure all 66 schema columns exist (fill missing ones with NaN)
for col in SCHEMA_COLS:
    if col not in df.columns:
        df[col] = np.nan
        print(f"  Added missing schema column: {col}")

# MIMIC-specific extra columns
MIMIC_EXTRAS = [
    "subject_id", "stay_id", "hadm_id",
    "ed_intime", "ed_outtime",
    "anchor_year_group", "year_offset",
    "arrival_year_real",
    "visit_number", "total_visits_patient",
]

# Missing flags (sorted)
missing_flag_cols = sorted([c for c in df.columns if c.endswith("_missing")])

# Build final column order
final_cols = SCHEMA_COLS + MIMIC_EXTRAS + missing_flag_cols

# Keep only columns that exist in df
final_cols = [c for c in final_cols if c in df.columns]

# Add any remaining columns not yet in the list (safety net)
remaining = [c for c in df.columns if c not in final_cols]
if remaining:
    print(f"  Extra columns not in final order (dropping): {remaining}")

df_final = df[final_cols].copy()

# --- Assert all 66 schema columns are present ---
missing_schema = [c for c in SCHEMA_COLS if c not in df_final.columns]
assert len(missing_schema) == 0, f"Missing schema columns: {missing_schema}"
print(f"✓ All {len(SCHEMA_COLS)} schema columns present")

print(f"Final shape: {df_final.shape}")
print(f"Total columns: {len(df_final.columns)} "
      f"(66 schema + {len(MIMIC_EXTRAS)} MIMIC extras + {len(missing_flag_cols)} flags)")


## Section 17 - QC & export

Final quality checks and export to `dataset/processed/final_mimic.csv`.


In [ ]:
print("=" * 60)
print("FINAL QC REPORT")
print("=" * 60)

# Shape
print(f"\n--- Shape ---")
print(f"Rows: {df_final.shape[0]:,}")
print(f"Columns: {df_final.shape[1]}")

# Missingness per column
print(f"\n--- Missingness per column ---")
miss = df_final.isna().mean().sort_values(ascending=False)
for col, pct in miss.items():
    if pct > 0:
        print(f"  {col}: {pct*100:.1f}%")

# All-NaN columns
all_nan_cols = [c for c in SCHEMA_COLS if df_final[c].isna().all()]
print(f"\n--- All-NaN schema columns ({len(all_nan_cols)}) ---")
expected_nan = {"transport_origin", "pain_location", "mental_status_triage",
                "triage_nurse_id", "gcs_total", "chief_complaint_system"}
for c in all_nan_cols:
    status = "✓ expected" if c in expected_nan else "⚠ UNEXPECTED"
    print(f"  {c} ({status})")

unexpected = set(all_nan_cols) - expected_nan
if unexpected:
    print(f"\n⚠ UNEXPECTED all-NaN columns: {unexpected}")
else:
    print(f"\n✓ All-NaN columns match expected list")

# Visit number distribution
print(f"\n--- Visit number distribution ---")
print(df_final["visit_number"].describe())

# Vital stats post-clipping
print(f"\n--- Vital signs summary (post-clipping) ---")
vital_cols = ["systolic_bp", "diastolic_bp", "heart_rate", "respiratory_rate",
              "temperature_c", "spo2", "pain_score", "shock_index", "news2_score"]
for v in vital_cols:
    if v in df_final.columns and df_final[v].notna().any():
        s = df_final[v]
        print(f"  {v}: n={s.notna().sum():,}, "
              f"mean={s.mean():.1f}, med={s.median():.1f}, "
              f"min={s.min():.1f}, max={s.max():.1f}")

# Disposition distribution
print(f"\n--- Disposition distribution ---")
if "disposition" in df_final.columns:
    print(df_final["disposition"].value_counts())

# Triage acuity distribution
print(f"\n--- Triage acuity distribution ---")
print(df_final["triage_acuity"].value_counts().sort_index())

# Export
out_path = OUT_DIR / "final_mimic.csv"
df_final.to_csv(out_path, index=False)
print(f"\n✓ Saved to {out_path}")
print(f"  File size: {out_path.stat().st_size / 1e6:.1f} MB")
